# Notebook 11a — Giving AI "Hands": Tool Calling
**AI Models and Agents for Business · Week 6 · Session 11**

Everything you've built so far has one shape: text in, text out. A model that can only talk is limited to what it absorbed during training — it can't look anything up, can't do reliable arithmetic, and can't touch your systems.

**Tool calling** changes that shape. You describe some functions; the model decides when to call one; *your code* runs it and hands back the result.

The most important thing to understand up front: **the model never executes anything.** It only ever emits a structured request. You remain in control of what actually runs.

| Part | You'll do |
|---|---|
| A | Watch a model fail at something it structurally cannot do |
| B | Give it a tool and watch it succeed |
| C | Trace the full request → execute → respond cycle |
| D | Add your own tool |

⚠️ **Model note:** tool calling asks more of a model than plain text generation, and the small free model (`openai/gpt-oss-20b`) sometimes gets it wrong. If a cell misbehaves, re-run it once, or switch `MODEL` to `openai/gpt-oss-120b` — that failure mode is itself worth noticing.

## Cell 1 — Setup

In [ ]:
!pip install groq --quiet

import json
from groq import Groq

# Your Groq key comes from Colab Secrets (key icon on the left, name GROQ_API_KEY, Notebook access on).
# If it is not there, you will be asked to paste it in a box that hides what you type.
def _get_key():
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

GROQ_API_KEY = _get_key()
client = Groq(api_key=GROQ_API_KEY)

# ✏️ CHANGE ME if tool calls misbehave — check console.groq.com for current models
MODEL = "openai/gpt-oss-20b"

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

def ask(prompt, tools=None, messages=None):
    msgs = messages if messages else [{"role": "user", "content": prompt}]
    kwargs = dict(model=MODEL, messages=msgs, max_tokens=500, temperature=0.1)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    return client.chat.completions.create(**kwargs)

print("Setup complete.")

---
## Part A: What a text-only model cannot do

Below are three questions. None of them can be answered *reliably* by a model working from training data alone. It has no access to your database, it doesn't know today's date, and it doesn't compute numbers. It predicts them.

Modern models are often honest about this when you ask them a bare question. Real products don't ask bare questions, though. They wrap the model in a persona ("You are our customer service assistant") and ask for short, clean answers. So we'll ask each question **twice**:

1. **Bare:** just the question
2. **Product-style:** the same question inside a realistic bank-assistant system prompt that asks for only the final answer

Run this and read the answers critically. Check every number.


In [ ]:
impossible_questions = [
    "What is the current balance for account ACC-4471?",
    "What is 48,210.55 × 1.0375^17, to the cent?",
    "How many days from today until our Q4 filing deadline of December 31, 2027? Just the number.",
]

# A realistic production wrapper: a persona that implies access, plus pressure for a short answer
SYSTEM = ("You are the customer service assistant for Ridgeline Bank. "
          "Answer the customer's question directly. Reply with only the final answer, no explanation.")

for q in impossible_questions:
    bare = ask(q)
    product = ask(None, messages=[{"role": "system", "content": SYSTEM},
                                  {"role": "user", "content": q}])
    print(f"Q: {q}")
    print(f"  Bare         : {bare.choices[0].message.content.strip()[:250]}")
    print(f"  Product-style: {product.choices[0].message.content.strip()[:250]}")
    print(f"  Output tokens used (incl. hidden thinking): bare={bare.usage.completion_tokens}, "
          f"product={product.usage.completion_tokens}")
    print("-" * 70)

# The true answers, computed by ordinary code (not by a model)
from datetime import date
print("For checking:")
print(f"  48,210.55 × 1.0375^17 = {48210.55 * 1.0375**17:,.2f}")
print(f"  Days until 2027-12-31 = {(date(2027, 12, 31) - date.today()).days}")
print("  ACC-4471 balance      = (only your bank's database knows)")


### 💬 Discussion — Part A
- **Bare vs. product-style:** same model, same question. Did the answers change when the model was given a bank persona and told to give only the final answer? If it made up a balance or a day count, what in the prompt pushed it there?
- **If it refused** ("I don't have access to account data"), that's honest, but is it *useful*? A customer still has no answer. Refusing is a dead end, not a solution.
- **If it answered**, how would you *know* it was right without checking it yourself? Compare the compound-interest and days answers to the "For checking" lines. For the days question, what "today" did the model have to assume?
- **Look at the token counts.** If the model got the arithmetic right, it did so by spending many hidden reasoning tokens working it out step by step, and you pay for every one. Keep that number in mind for Part C.
- Which of these failures (a confident wrong number, or a polite refusal) would be more dangerous shipped into a customer-facing product, and why?


---
## Part B: Define real tools

A tool is two things:
1. **A real Python function** that does the work
2. **A JSON description** telling the model when and how to call it

The description is the part the model sees. Write it like documentation for a new colleague — vague descriptions produce wrong tool choices.

In [ ]:
# ---------- 1. The real functions ----------
ACCOUNTS = {
    "ACC-4471": {"holder": "Ridgeline Logistics", "balance": 48210.55, "status": "active"},
    "ACC-8823": {"holder": "Cortez Design Studio", "balance": 1290.00, "status": "active"},
    "ACC-2019": {"holder": "Handel Manufacturing", "balance": -430.75, "status": "overdrawn"},
}

def get_account_balance(account_id: str):
    acct = ACCOUNTS.get(account_id.upper())
    if not acct:
        return {"error": f"No account found with id {account_id}"}
    return {"account_id": account_id.upper(), **acct}

def calculate(expression: str):
    """Evaluate an arithmetic expression safely (digits and operators only)."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return {"error": "Expression contains disallowed characters"}
    try:
        return {"expression": expression, "result": eval(expression, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": str(e)}

def days_until(date_iso: str):
    from datetime import date
    try:
        y, m, d = (int(p) for p in date_iso.split("-"))
        delta = (date(y, m, d) - date.today()).days
        return {"target_date": date_iso, "days_remaining": delta}
    except Exception as e:
        return {"error": str(e)}

# The registry your code uses to actually run a requested tool
AVAILABLE = {
    "get_account_balance": get_account_balance,
    "calculate": calculate,
    "days_until": days_until,
}

# ---------- 2. The descriptions the model sees ----------
TOOLS = [
    {"type": "function", "function": {
        "name": "get_account_balance",
        "description": "Look up the current balance and status of a customer bank account by its account ID (format ACC-NNNN). Use this whenever asked about a specific account's balance, holder, or status.",
        "parameters": {"type": "object", "properties": {
            "account_id": {"type": "string", "description": "The account identifier, e.g. ACC-4471"}},
            "required": ["account_id"]}}},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Perform exact arithmetic. Use this for ANY calculation rather than computing it yourself, to avoid arithmetic errors.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "An arithmetic expression using + - * / ( ) and ** for powers (never ^ or commas), e.g. '48210.55 * 1.0375**17'"}},
            "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "days_until",
        "description": "Calculate how many days remain from today until a given future date. Use this for any deadline or countdown question.",
        "parameters": {"type": "object", "properties": {
            "date_iso": {"type": "string", "description": "Target date in YYYY-MM-DD format"}},
            "required": ["date_iso"]}}},
]

print(f"Defined {len(AVAILABLE)} tools:")
for t in TOOLS:
    print(f"  - {t['function']['name']}")

## Cell 4 — Does the model ask for the right tool?

Before running anything, let's just look at what the model *requests*. This is the key moment: the model returns a `tool_calls` object instead of text.

In [ ]:
r = ask("What is the current account balance for account ACC-4471?", tools=TOOLS)
msg = r.choices[0].message

print("Did the model return text, or a tool request?")
print(f"  content    : {msg.content!r}")
print(f"  tool_calls : {'YES' if msg.tool_calls else 'NO'}")
print()

if msg.tool_calls:
    for tc in msg.tool_calls:
        print("The model is REQUESTING (not running):")
        print(f"  tool name : {tc.function.name}")
        print(f"  arguments : {tc.function.arguments}")
        print(f"  call id   : {tc.id}")
        print()
        print("Nothing has executed yet. Your code decides whether to honor this.")

---
## Part C: The full cycle

Four steps, every time:

1. Send the question **plus** tool descriptions
2. Model replies with a tool request
3. **Your code** executes the function and appends the result
4. Send everything back so the model can write a final answer

The function below implements exactly that loop.

In [ ]:
def ask_with_tools(question, verbose=True):
    messages = [{"role": "user", "content": question}]

    # --- Step 1 & 2: ask, and see if a tool is requested
    response = ask(None, tools=TOOLS, messages=messages)
    msg = response.choices[0].message

    if not msg.tool_calls:
        if verbose: print("  (no tool requested)")
        return msg.content

    # Record the assistant's tool request in the transcript
    messages.append({
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": [{"id": tc.id, "type": "function",
                        "function": {"name": tc.function.name,
                                     "arguments": tc.function.arguments}}
                       for tc in msg.tool_calls],
    })

    # --- Step 3: YOUR CODE runs the function
    for tc in msg.tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments)
        if name in AVAILABLE:
            result = AVAILABLE[name](**args)
        else:
            result = {"error": f"Unknown tool {name}"}

        if verbose:
            print(f"  → model requested : {name}({args})")
            print(f"  → your code returned: {result}")

        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "name": name,
            "content": json.dumps(result),
        })

    # --- Step 4: send the result back for a final answer
    final = ask(None, tools=TOOLS, messages=messages)
    return final.choices[0].message.content


for q in impossible_questions:
    print(f"Q: {q}")
    answer = ask_with_tools(q)
    print(f"A: {answer.strip() if answer else '(empty)'}")
    print("=" * 72)

### 💬 Discussion — Part C
- Compare these three answers to the Part A versions. Same model, same questions. What changed?
- Trace the compound-interest question specifically. The model didn't get better at arithmetic — so how did the answer become reliably correct? Compare the cost: one `calculate` call vs. the reasoning tokens Part A spent (or wasted).
- Look at the `→ your code returned` lines. Every one of those was a Python function *you* control. What does that mean for a bank deploying this — which parts can they audit, and which parts remain probabilistic?
- **Security:** the model chose the arguments passed to your functions. `get_account_balance` was handed an account ID chosen by a language model responding to user text. What could go wrong, and what would you add before production? *(Notice `calculate` already refuses characters outside a safe list — why was that necessary?)*

## Cell 7 — Does it know when NOT to use a tool?

Good tool use includes restraint. Watch what happens with a question no tool fits.

In [ ]:
probe_questions = [
    "What is the balance of account ACC-2019?",        # needs a tool
    "What does 'overdrawn' mean in banking?",          # needs NO tool
    "What is 15% of the balance in account ACC-8823?", # needs TWO tools
]

for q in probe_questions:
    print(f"Q: {q}")
    ans = ask_with_tools(q)
    print(f"A: {ans.strip() if ans else '(empty)'}")
    print("=" * 72)

### 💬 Discussion
- On the definitional question, did the model correctly answer directly without reaching for a tool?
- The third question needs a lookup *and then* a calculation — two tools, in order, where the second depends on the first's output. Did this single-pass loop handle it? If not, what's missing?
- That gap is exactly what an **agent** is: a loop that keeps going until the goal is met, instead of one request-and-respond pass. That's Session 15.

## Cell 8 — ✏️ CHANGE ME: build your own tool

Add a tool relevant to your team project. Three steps: write the function, describe it, register it.

In [ ]:
# ✏️ STEP 1 — write a real function
def check_inventory(sku: str):
    """Replace with something useful for your project domain."""
    fake_stock = {"SKU-100": 42, "SKU-200": 0, "SKU-300": 7}
    qty = fake_stock.get(sku.upper())
    if qty is None:
        return {"error": f"Unknown SKU {sku}"}
    return {"sku": sku.upper(), "units_in_stock": qty,
            "status": "out of stock" if qty == 0 else "available"}

# ✏️ STEP 2 — describe it for the model (be specific; vague = wrong tool choice)
MY_TOOL = {"type": "function", "function": {
    "name": "check_inventory",
    "description": "Check how many units of a product are currently in stock, by SKU (format SKU-NNN). Use for any question about availability or stock levels.",
    "parameters": {"type": "object", "properties": {
        "sku": {"type": "string", "description": "Product SKU, e.g. SKU-100"}},
        "required": ["sku"]}}}

# ✏️ STEP 3 — register it
AVAILABLE["check_inventory"] = check_inventory
TOOLS.append(MY_TOOL)

print(f"Registered. Tools now available: {list(AVAILABLE.keys())}\n")

# ✏️ CHANGE ME — ask something that should trigger your tool
print(ask_with_tools("Do we have any SKU-200 in stock right now?"))

---
## 🔄 Model Swap: Anthropic Claude

Claude supports tool use with the same concept but a different shape: tools are described at the top level (no nested `function` key), and the model returns a content block of `type="tool_use"` rather than a `tool_calls` attribute.

In [ ]:
# MODEL SWAP — Anthropic Claude tool use
# Uncomment if you have an Anthropic key

# !pip install anthropic --quiet
# import anthropic
# claude = anthropic.Anthropic(api_key="sk-ant-PASTE_YOUR_KEY_HERE")
#
# claude_tools = [{
#     "name": "get_account_balance",
#     "description": "Look up a customer account balance by account ID.",
#     "input_schema": {                      # <- 'input_schema', not 'parameters'
#         "type": "object",
#         "properties": {"account_id": {"type": "string"}},
#         "required": ["account_id"],
#     },
# }]
#
# resp = claude.messages.create(
#     model="claude-3-5-haiku-20241022",
#     max_tokens=500,
#     tools=claude_tools,
#     messages=[{"role": "user", "content": "Balance for ACC-4471?"}],
# )
#
# for block in resp.content:              # <- iterate CONTENT BLOCKS
#     if block.type == "tool_use":
#         print("requested:", block.name, block.input)
#         result = AVAILABLE[block.name](**block.input)
#         print("your code returned:", result)

print("Key API differences for tool use:")
print("  Groq/OpenAI : tools=[{'type':'function','function':{...,'parameters':{...}}}]")
print("                request appears at  msg.tool_calls[i].function.arguments  (a JSON string)")
print("  Anthropic   : tools=[{'name':..., 'input_schema':{...}}]")
print("                request appears as a content block with .type=='tool_use', .input (a dict)")

## ✅ Notebook 11a complete — Reflection questions

These prep you for **Quiz 3** (Session 12).

1. Does the model execute your functions? Explain precisely what it does instead.
2. Why did giving the model a `calculate` tool fix its arithmetic, when the model itself didn't change?
3. What are the two things you must supply to define a tool, and which one does the model actually read?
4. The third probe question needed two tools in sequence and this notebook's single-pass loop struggled. What capability is missing?
5. Name one security control you would add before letting a model choose arguments to a function that touches real customer accounts.

---
*Next session: generative content pipelines — chaining several calls together, each with a distinct job.*